# Raster Validation

Validates raster built-up datasets (TEMPO, GHSL Built-S/V/H, Google OBT) against reference building footprints.

Pipeline per city:
1. Tile the AOI into 1 km × 1 km cells
2. For each enabled raster candidate, read the year-specific file (`{city_slug}_{name}_{year}.tif`)
3. Rasterize reference footprints onto the candidate grid (fractional coverage via oversampling)
4. Compute tile-level binary (TP/FP/FN/F1) and area-based metrics
5. Save tile metrics, city summary, and figures to `outputs/`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"

# Set overwrite=True to re-run cities whose raster outputs already exist.
# When False (default), cities with an existing sentinel parquet are skipped automatically.
OVERWRITE = True

In [3]:
import sys, os
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
# import rasterio
# from pathlib import Path

# tile_path = Path(f"{PROJECT_ROOT}/data/WSFsoftRelease/Oceania/WSFtracker_20160701-20250701_-126_-26.tif")

# with rasterio.open(tile_path) as src:
#     print("CRS:", src.crs)
#     print("Bounds:", src.bounds)
#     print("Shape:", (src.height, src.width))
#     print("Resolution:", src.res)

In [5]:
# from glob import glob

# paths = glob(f"{PROJECT_ROOT}/data/WSFsoftRelease/**/*.tif", recursive=True)[:5]

# for p in paths:
#     with rasterio.open(p) as src:
#         print(p.split("/")[-1], "->", src.bounds)

In [6]:
import logging
import yaml
from src.validator import UrbanValidator

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

with open(CONFIG_PATH) as f:
    _cfg_preview = yaml.safe_load(f)

raster_datasets = _cfg_preview.get("raster", {}).get("datasets", [])
enabled = [
    f"{d['name']}" + (f"_{d['year']}" if d.get("year") is not None else "")
    for d in raster_datasets
    if d.get("enabled", True)
]
print(f"Config: {CONFIG_PATH}")
print(f"Enabled raster datasets: {enabled}")

Config: /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml
Enabled raster datasets: ['obt_2023', 'tempo_2023q4', 'ghsl_built_s_2020', 'wsf_tracker']


In [7]:
# Patch overwrite flag and instantiate the Validator.
# This reads the config and AOI tracker, resolves file paths,
# and logs how many city datasets are queued.
import tempfile

with open(CONFIG_PATH) as f:
    _cfg_patched = yaml.safe_load(f)
_cfg_patched.setdefault("output", {})["overwrite"] = OVERWRITE

_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", delete=False, dir=PROJECT_ROOT / "configs"
)
yaml.dump(_cfg_patched, _tmp)
_tmp.close()
_PATCHED_CONFIG_PATH = _tmp.name

v = UrbanValidator(_PATCHED_CONFIG_PATH)
print(f"\nCities queued: {len(v.datasets)}")
for ds in v.datasets:
    print(f"  {ds['id']}")

20:32:09  INFO      Validation tracker: 85 -> 85 suitable rows.
20:33:28  INFO      Loaded 85 dataset(s) for validation.
20:33:28  INFO      Loaded 85 dataset(s) for validation.



Cities queued: 85
  afg-ayabak
  afg-bamyan
  afg-bazarak
  afg-charikar
  afg-east-kabul
  afg-ferozkoh
  afg-gardez
  afg-herat
  afg-jalalaba
  afg-mahmud-e-raqi
  afg-mazar-e-sharif
  afg-poruns
  afg-qala-e-naw
  afg-west-kabul
  afg-zaranj
  ant-curacao
  bgd-rohingya
  blz-burrell-boom
  bra-nova-sussuarana
  col-san-antonio-de-prado
  cvg-saint-vincent-grenadines
  dom-dominica
  gha-accra
  gha-aiyim-sraha
  gha-dansoman
  gha-nawuni
  gha-sawla-tuna
  gha-wa
  grd-grenada
  jam-kingston
  jam-saint-catherine
  jpn-ashiya-hama
  jpn-hiroshima
  jpn-iwate
  jpn-izu-oshima
  jpn-kashima
  jpn-numakunai
  ken-kakuma
  ken-kakuma-kalobeyei
  ken-mukuru
  ken-nairobi
  lbr-monrovia
  lby-almarj
  lby-bayda
  lby-darnah
  lby-susah
  lca-saint-lucia
  maf-saint-martin
  mmr-patheingyi-mandalay
  moz-de-maio
  moz-djonasse
  mwi-lilongwe
  mwi-mlowe
  ner-niame
  nga-ibadan
  phl-bagamanoc
  phl-barangay
  phl-catanduanes
  phl-juraojurao-anini-y-antique
  phl-pasig
  phl-poblacion-

In [8]:
# Preview: show which raster files will be looked up for each city × dataset combination.
# Files that don't exist on disk are flagged so you can fix paths before running.
import pandas as pd
from pathlib import Path

data_dir = Path(_cfg_patched.get("root_dir", str(PROJECT_ROOT))) / _cfg_patched.get("data_dir", "data/01_raw")

preview_rows = []
for ds in v.datasets:
    city_slug    = ds["id"].lower()
    city_slug_us = city_slug.replace("-", "_")
    rast_dir     = data_dir / ds["id"] / "raster"

    for cand in _cfg_patched.get("raster", {}).get("datasets", []):
        if not cand.get("enabled", True):
            continue
        ds_name = cand["name"].replace("-", "_")
        year    = cand.get("year")
        if year is not None:
            fpath = rast_dir / f"{city_slug_us}_{ds_name}_{year}.tif"
        else:
            matches = sorted(rast_dir.glob(f"{city_slug_us}_{ds_name}*.tif"))
            fpath   = matches[0] if matches else rast_dir / f"{city_slug_us}_{ds_name}_?.tif"
        preview_rows.append({
            "city":    ds["id"],
            "dataset": f"{ds_name}_{year}" if year is not None else ds_name,
            "file":    fpath.name,
            "exists":  fpath.exists(),
        })

preview_df = pd.DataFrame(preview_rows)
missing = preview_df[~preview_df["exists"]]
print(f"Total city × dataset pairs: {len(preview_df)}")
print(f"Missing files:              {len(missing)}")
display(preview_df)

Total city × dataset pairs: 340
Missing files:              20


,city,dataset,file,exists
0,afg-ayabak,obt_2023,afg_ayabak_obt_2023.tif,True
1,afg-ayabak,tempo_2023q4,afg_ayabak_tempo_2023q4.tif,True
2,afg-ayabak,ghsl_built_s_2020,afg_ayabak_ghsl_built_s_2020.tif,True
3,afg-ayabak,wsf_tracker,afg_ayabak_wsf_tracker.tif,True
4,afg-bamyan,obt_2023,afg_bamyan_obt_2023.tif,True
...,...,...,...,...
335,uga-nakamiro,wsf_tracker,uga_nakamiro_wsf_tracker.tif,True
336,ukr-pulyny,obt_2023,ukr_pulyny_obt_2023.tif,False
337,ukr-pulyny,tempo_2023q4,ukr_pulyny_tempo_2023q4.tif,True
338,ukr-pulyny,ghsl_built_s_2020,ukr_pulyny_ghsl_built_s_2020.tif,True


In [9]:
results = v.validate_raster()

# Clean up temp config
try:
    os.unlink(_PATCHED_CONFIG_PATH)
except Exception:
    pass

# Summary
summary = pd.DataFrame(
    [{"city": k, "status": "ok" if ok else "failed"} for k, ok in results.items()]
)
print(f"\nDone — {len(summary)} cities processed.\n")
display(summary.groupby("status")["city"].count().rename("count").to_frame())
display(summary)

20:33:52  INFO      ━━━━  afg-ayabak (raster)  ━━━━
20:33:52  INFO      MEM [afg-ayabak raster start] RSS = 356 MB
20:33:52  INFO      [afg-ayabak] Raster CRS: EPSG:32642 | 51 tiles.
20:33:54  INFO      [afg-ayabak] Raster reference buildings: 18702 (from 1 file(s))
20:33:54  INFO      [afg-ayabak / obt_2023] Raster candidate: afg_ayabak_obt_2016.tif
20:34:04  INFO      [afg-ayabak / obt_2023] Raster tile metrics saved → raster_metrics_tiles_obt_2023.parquet
20:34:04  INFO      MEM [afg-ayabak/obt_2023 raster done] RSS = 503 MB
20:34:04  INFO      [afg-ayabak / tempo_2023q4] Raster candidate: afg_ayabak_tempo_2023q4.tif
20:34:11  INFO      [afg-ayabak / tempo_2023q4] Raster tile metrics saved → raster_metrics_tiles_tempo_2023q4.parquet
20:34:11  INFO      MEM [afg-ayabak/tempo_2023q4 raster done] RSS = 513 MB
20:34:11  INFO      [afg-ayabak / ghsl_built_s_2020] Raster candidate: afg_ayabak_ghsl_built_s_1975.tif
20:34:17  INFO      [afg-ayabak / ghsl_built_s_2020] Raster tile metrics sa

[herat_WGB.geojson] fixing 1 invalid geometries...


20:45:16  INFO      [afg-herat] Raster reference buildings: 162977 (from 1 file(s))
20:45:16  WARNING   [afg-herat / obt] No raster file found (pattern: afg_herat_obt*).
20:45:16  INFO      [afg-herat / tempo_2023q4] Raster candidate: afg_herat_tempo_2023q4.tif
20:46:07  INFO      [afg-herat / tempo_2023q4] Raster tile metrics saved → raster_metrics_tiles_tempo_2023q4.parquet
20:46:07  INFO      MEM [afg-herat/tempo_2023q4 raster done] RSS = 1807 MB
20:46:07  INFO      [afg-herat / ghsl_built_s_2020] Raster candidate: afg_herat_ghsl_built_s_1975.tif
20:46:58  INFO      [afg-herat / ghsl_built_s_2020] Raster tile metrics saved → raster_metrics_tiles_ghsl_built_s_2020.parquet
20:46:58  INFO      MEM [afg-herat/ghsl_built_s_2020 raster done] RSS = 1807 MB
20:46:58  INFO      [afg-herat / wsf_tracker] Raster candidate: afg_herat_wsf_tracker.tif
20:47:49  INFO      [afg-herat / wsf_tracker] Raster tile metrics saved → raster_metrics_tiles_wsf_tracker.parquet
20:47:49  INFO      MEM [afg-her

[mazar-e-sharif_WGB.geojson] fixing 1 invalid geometries...


20:50:58  INFO      [afg-mazar-e-sharif] Raster reference buildings: 127820 (from 1 file(s))
20:50:58  WARNING   [afg-mazar-e-sharif / obt] No raster file found (pattern: afg_mazar_e_sharif_obt*).
20:50:58  INFO      [afg-mazar-e-sharif / tempo_2023q4] Raster candidate: afg_mazar_e_sharif_tempo_2023q4.tif
20:51:36  INFO      [afg-mazar-e-sharif / tempo_2023q4] Raster tile metrics saved → raster_metrics_tiles_tempo_2023q4.parquet
20:51:36  INFO      MEM [afg-mazar-e-sharif/tempo_2023q4 raster done] RSS = 1832 MB
20:51:36  INFO      [afg-mazar-e-sharif / ghsl_built_s_2020] Raster candidate: afg_mazar_e_sharif_ghsl_built_s_1975.tif
20:52:12  INFO      [afg-mazar-e-sharif / ghsl_built_s_2020] Raster tile metrics saved → raster_metrics_tiles_ghsl_built_s_2020.parquet
20:52:12  INFO      MEM [afg-mazar-e-sharif/ghsl_built_s_2020 raster done] RSS = 1832 MB
20:52:12  INFO      [afg-mazar-e-sharif / wsf_tracker] Raster candidate: afg_mazar_e_sharif_wsf_tracker.tif
20:52:51  INFO      [afg-mazar

...

[kolleh_hotosm.geojson] Dropped 2 null/empty geometries (576 → 574)


22:16:47  INFO      [sle-kolleh / obt_2023] Raster tile metrics saved → raster_metrics_tiles_obt_2023.parquet
22:16:47  INFO      MEM [sle-kolleh/obt_2023 raster done] RSS = 4589 MB
22:16:47  INFO      [sle-kolleh / tempo_2023q4] Raster candidate: sle_kolleh_tempo_2023q4.tif
22:16:48  INFO      [sle-kolleh / tempo_2023q4] Raster tile metrics saved → raster_metrics_tiles_tempo_2023q4.parquet
22:16:48  INFO      MEM [sle-kolleh/tempo_2023q4 raster done] RSS = 4589 MB
22:16:48  INFO      [sle-kolleh / ghsl_built_s_2020] Raster candidate: sle_kolleh_ghsl_built_s_1975.tif
22:16:48  INFO      [sle-kolleh / ghsl_built_s_2020] Raster tile metrics saved → raster_metrics_tiles_ghsl_built_s_2020.parquet
22:16:48  INFO      MEM [sle-kolleh/ghsl_built_s_2020 raster done] RSS = 4589 MB
22:16:48  INFO      [sle-kolleh / wsf_tracker] Raster candidate: sle_kolleh_wsf_tracker.tif
22:16:49  INFO      [sle-kolleh / wsf_tracker] Raster tile metrics saved → raster_metrics_tiles_wsf_tracker.parquet
22:16:49 


Done — 85 cities processed.



,count
status,
failed,3
ok,82


,city,status
0,afg-ayabak,ok
1,afg-bamyan,ok
2,afg-bazarak,ok
3,afg-charikar,ok
4,afg-east-kabul,ok
...,...,...
80,uga-bugoye,ok
81,uga-kampala,ok
82,uga-kanara,ok
83,uga-nakamiro,ok


In [11]:
summary

,city,status
0,afg-ayabak,ok
1,afg-bamyan,ok
2,afg-bazarak,ok
3,afg-charikar,ok
4,afg-east-kabul,ok
...,...,...
80,uga-bugoye,ok
81,uga-kampala,ok
82,uga-kanara,ok
83,uga-nakamiro,ok
